<a href="https://colab.research.google.com/github/k9Sx3CC/01_first_look_and_discovery.ipynb/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k9Sx3CC/flyrank-internship-test/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I created a simple baseline action score that prioritizes pages based on signals available before any future outcome is known. The score increases when a page has low engagement, poor average ranking, older content, or low click-through rate. These signals may indicate that a page could benefit from manual review. The score is intended for decision support rather than making automatic content decisions.
Reason codes:

- LOW_ENGAGEMENT i.e., Engagement rate is zero.
- LOW_CTR i.e., Click-through rate is below the dataset median.
- POOR_POSITION i.e., Average search position is worse than the dataset median.
- OLD_CONTENT i.e., Content age is above the dataset median.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
import pandas as pd
import numpy as np
import os
import subprocess

# Clone repository if needed
REPO_DIR = "flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into repository
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Remove columns that could cause leakage
drop_cols = [
    "trend_direction",
    "trend_pct",
    "provider_used",
    "model_used"
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Calculate thresholds
ctr_med = df["ctr"].median()
position_med = df["avg_position"].median()
age_med = df["content_age_days"].median()

# Initialize score and reason code
df["baseline_score"] = 0
df["reason_code"] = ""

# Rule 1: Zero engagement
mask = df["engagement_rate"] == 0
df.loc[mask, "baseline_score"] += 2
df.loc[mask, "reason_code"] += "LOW_ENGAGEMENT;"

# Rule 2: Low CTR
mask = df["ctr"] < ctr_med
df.loc[mask, "baseline_score"] += 1
df.loc[mask, "reason_code"] += "LOW_CTR;"

# Rule 3: Poor search position
mask = df["avg_position"] > position_med
df.loc[mask, "baseline_score"] += 1
df.loc[mask, "reason_code"] += "POOR_POSITION;"

# Rule 4: Older content
mask = df["content_age_days"] > age_med
df.loc[mask, "baseline_score"] += 1
df.loc[mask, "reason_code"] += "OLD_CONTENT;"

# Remove the trailing semicolon from reason codes
df["reason_code"] = df["reason_code"].str.rstrip(";")

# Rank pages
df = df.sort_values(
    by="baseline_score",
    ascending=False
)

# Save output
os.makedirs("work/outputs", exist_ok=True)

output_file = "work/outputs/baseline_action_score.csv"
df.to_csv(output_file, index=False)

print("Saved to:", output_file)

# Display Top 20 pages
df[[
    "content_id",
    "baseline_score",
    "reason_code"
]].head(20)

Saved to: work/outputs/baseline_action_score.csv


,content_id,baseline_score,reason_code
30,content_249298388b45,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT
1,content_a1fb4e703a9e,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT
11709,content_2a0d93e5fed8,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT
11705,content_9cb43a654c9b,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT
11699,content_51de95d5b26b,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT
11727,content_0260224a5a30,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT
11726,content_291c000503bf,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT
11722,content_6efb8fa48ebe,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT
11721,content_6e2368caa550,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT
19742,content_72348d55e12c,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The highest-ranked pages generally matched multiple rule conditions, especially zero engagement combined with poor ranking or low CTR. These pages were assigned a higher priority because several observed signals suggested they may benefit from review. Confidence: Medium.

These recommendations can be wrong because

- Some pages may target low-volume or niche queries.
- Seasonal content may naturally have lower traffic.
- Some pages may intentionally have low engagement despite meeting business goals.
- The rule is intended for decision support and should not replace manual review.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df.head(20)

review = top20[[
    "content_id",
    "baseline_score",
    "reason_code"
]].copy()

review["action"] = "Review"

review["confidence_note"] = np.where(
    review["baseline_score"] >= 5,
    "Higher confidence",
    "Medium confidence"
)


review["what_would_make_it_wrong"] = (
    "Seasonal topic, niche audience, or intentional low engagement."
)

review

,content_id,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
30,content_249298388b45,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT,Review,Higher confidence,"Seasonal topic, niche audience, or intentional..."
1,content_a1fb4e703a9e,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT,Review,Higher confidence,"Seasonal topic, niche audience, or intentional..."
11709,content_2a0d93e5fed8,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT,Review,Higher confidence,"Seasonal topic, niche audience, or intentional..."
11705,content_9cb43a654c9b,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT,Review,Higher confidence,"Seasonal topic, niche audience, or intentional..."
11699,content_51de95d5b26b,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT,Review,Higher confidence,"Seasonal topic, niche audience, or intentional..."
11727,content_0260224a5a30,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT,Review,Higher confidence,"Seasonal topic, niche audience, or intentional..."
11726,content_291c000503bf,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT,Review,Higher confidence,"Seasonal topic, niche audience, or intentional..."
11722,content_6efb8fa48ebe,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT,Review,Higher confidence,"Seasonal topic, niche audience, or intentional..."
11721,content_6e2368caa550,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT,Review,Higher confidence,"Seasonal topic, niche audience, or intentional..."
19742,content_72348d55e12c,5,LOW_ENGAGEMENT;LOW_CTR;POOR_POSITION;OLD_CONTENT,Review,Higher confidence,"Seasonal topic, niche audience, or intentional..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some high-ranked pages may be false positives because rule-based scoring does not capture business context such as seasonal demand, niche audiences, or strategic content. Therefore, the ranked queue should be used to support manual review rather than automatically refreshing content.

I confirmed that no future outcome variables (such as trend_direction or trend_pct) and no product-generated recommendation fields were used when constructing the baseline score. The rule relies only on information available before prediction.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_cols = [
    "trend_direction",
    "trend_pct",
    "provider_used",
    "model_used"
]

present = [c for c in leakage_cols if c in df.columns]

print("Leakage columns remaining:")
print(present)

Leakage columns remaining:
[]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.